# UAE Healthcare Prediction — reproducible Colab notebook
**Goal:** compare basic measurements with additional laboratory biomarkers for identifying current abnormal glycemic status in 555 young Emirati women.

**Run:** Open with Google Colaboratory, use a CPU runtime, then **Runtime → Run all**. Authorize Google Drive access when prompted. Files are loaded from the project folder; outputs are saved into a new timestamped `runs` subfolder. No GPU needed.

This notebook runs six classifiers plus a prevalence baseline. Fixed pilot configurations, no tuning. It does not establish future diabetes prediction or clinical readiness. The added methods have no preclaimed results.

Source: https://doi.org/10.1371/journal.pone.0252884 ; dataset: https://doi.org/10.6084/m9.figshare.14547246.v1 . Verify reuse terms before redistribution/publication.

## 1. Install the SPSS reader

In [ ]:
%pip -q install pyreadstat==1.3.6

## 2. Connect Google Drive and choose settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime
PROJECT = Path('/content/drive/MyDrive/UAE Healthcare Prediction - Colab')
assert (PROJECT/'S1_File.sav').exists(), 'Check PROJECT or mount the Google account containing the folder.'
SEEDS = [2026, 2027, 2028]  # Set [2026] for one faster exploratory repetition.
RUN_ORIGINAL_SENSITIVITY = True
RUN_DIR = PROJECT/'runs'/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Results folder:', RUN_DIR)

## 3. Imports

In [ ]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
import pyreadstat, sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, confusion_matrix, brier_score_loss

from types import SimpleNamespace
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay
from sklearn.calibration import CalibrationDisplay


## 4. Load and audit the data
The notebook regenerates the cleaned analysis from the original SPSS file so every change is reproducible. `cleaned_analysis.csv` in the project folder is a ready-made copy of the selected inputs and target.

BMI is derived from height and weight for every record. Four supplied BMI values differ by >1. One DBP value (6.1) is set missing under a conservative DBP <20 flag and imputed **inside each training fold**. Other unusual measurements remain unchanged; source verification is still needed. Fat percentage is not used. Existing categorical labels remain unchanged and must not be used as features.

Target `Ab.Gly`: 370 normal, 185 abnormal. Glucose, HbA1c, their categories and Diabetes are excluded to prevent circular prediction. `Relative` indicates parental diabetes **or hypertension**.

In [ ]:
a=SimpleNamespace(input=str(PROJECT/'S1_File.sav'),out=str(RUN_DIR))
out=Path(a.out);out.mkdir(parents=True,exist_ok=True)
d,meta=pyreadstat.read_sav(a.input);y=d['Ab.Gly'].astype(int).to_numpy()
assert len(d)==555 and d.Code.nunique()==555
assert np.array_equal(y,(d.Diabetes>0).astype(int))
basic=['Age','Relative','BMI','WC','SBP','DBP']
extended=basic+['Trigl','Chol','LDL','HDL','Hs.CRP','Hb']
clean=d.copy();clean['BMI']=d.Weight/(d.Height/100)**2
bad_bp=d.DBP<20 # conservative flag for a grossly questionable value; not a clinical cutoff
clean.loc[bad_bp,'DBP']=np.nan
changes=[]
for i in d.index[abs(clean.BMI-d.BMI)>1]:
 changes.append({'row':int(i+1),'variable':'BMI','original':d.loc[i,'BMI'],'analysis_value':clean.loc[i,'BMI'],'reason':'Difference >1 from height/weight-derived BMI; derived BMI used for every participant'})
for i in d.index[bad_bp]:changes.append({'row':int(i+1),'variable':'DBP','original':d.loc[i,'DBP'],'analysis_value':None,'reason':'DBP <20 flagged; training-fold median imputation'})
pd.DataFrame(changes).to_csv(out/'audit_flags.csv',index=False)
clean[basic+extended[len(basic):]+['Ab.Gly']].to_csv(out/'analysis_data.csv',index=False)
pd.DataFrame([{'variable':c,'label':meta.column_names_to_labels[c],'missing':int(d[c].isna().sum()),'unique':int(d[c].nunique())} for c in d]).to_csv(out/'data_dictionary.csv',index=False)

print('Shape:',d.shape)
print(d['Ab.Gly'].value_counts())
display(pd.DataFrame(changes))

## 5. Methods
All configurations are fixed before evaluation. Six methods plus a baseline; identical splits for basic and extended inputs. Adding or tuning models after inspecting these results remains exploratory.

In [ ]:
models={
 'Prevalence baseline':lambda:DummyClassifier(strategy='prior'),
 'Logistic regression':lambda:LogisticRegression(C=1,max_iter=2000),
 'Random forest':lambda:RandomForestClassifier(n_estimators=300,min_samples_leaf=5,max_features='sqrt',random_state=2026,n_jobs=1),
 'Gradient boosting':lambda:GradientBoostingClassifier(n_estimators=100,learning_rate=.05,max_depth=2,min_samples_leaf=10,random_state=2026)}

models.update({
 'SVM (RBF)': lambda:SVC(C=1, kernel='rbf', probability=True, random_state=2026),
 'K-nearest neighbours':lambda:KNeighborsClassifier(n_neighbors=15),
 'Extra trees':lambda:ExtraTreesClassifier(n_estimators=300,min_samples_leaf=5,random_state=2026,n_jobs=1)
})
print('Methods:', list(models))


## 6. Repeated five-fold cross-validation
Training-fold median imputation and scaling only. Default threshold 0.5. No SMOTE, feature selection or hyperparameter tuning. Metrics are computed from held-out predictions for each repetition, then averaged. Repetitions are not independent test cohorts or confidence intervals. SVM probability estimation is fitted using training data only.

In [ ]:
rows=[];predrows=[]
versions = [('cleaned', clean)] + ([('original', d)] if RUN_ORIGINAL_SENSITIVITY else [])
for version,frame in versions:
 for feature_set,cols in [('Basic',basic),('Extended',extended)]:
  for name,factory in models.items():
   for repeat,seed in enumerate(SEEDS,1):
    probs=np.zeros(len(d));folds=np.zeros(len(d),int)
    for fold,(tr,te) in enumerate(StratifiedKFold(5,shuffle=True,random_state=seed).split(frame,y),1):
     model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),factory())
     model.fit(frame.iloc[tr][cols],y[tr]);probs[te]=model.predict_proba(frame.iloc[te][cols])[:,1];folds[te]=fold
    pred=probs>=.5;tn,fp,fn,tp=confusion_matrix(y,pred).ravel()
    rows.append(dict(version=version,features=feature_set,model=name,repeat=repeat,AUC=roc_auc_score(y,probs),AP=average_precision_score(y,probs),accuracy=accuracy_score(y,pred),sensitivity=tp/(tp+fn),specificity=tn/(tn+fp),Brier=brier_score_loss(y,probs)))
    predrows.extend(dict(version=version,features=feature_set,model=name,repeat=repeat,row=i+1,fold=int(folds[i]),target=int(y[i]),probability=probs[i]) for i in range(len(y)))
   print(version,feature_set,name,flush=True)
r=pd.DataFrame(rows);r.to_csv(out/'metrics_by_repeat.csv',index=False)
pd.DataFrame(predrows).to_csv(out/'out_of_fold_predictions.csv',index=False)
summary=r.groupby(['version','features','model'])[['AUC','AP','accuracy','sensitivity','specificity','Brier']].mean().reset_index();summary.to_csv(out/'summary.csv',index=False)
(out/'run_metadata.json').write_text(json.dumps({'sha256':hashlib.sha256(Path(a.input).read_bytes()).hexdigest(),'python':platform.python_version(),'sklearn':sklearn.__version__,'pandas':pd.__version__,'numpy':np.__version__,'pyreadstat':pyreadstat.__version__,'basic':basic,'extended':extended,'seeds':SEEDS,'threshold':.5,'tuning':'None; fixed pilot configurations'},indent=2))
print(summary.to_string(index=False))


## 7. Compare discrimination, sensitivity and calibration

In [ ]:
display(summary[summary.version=='cleaned'].round(3))
primary = summary[(summary.version=='cleaned') & (summary.model!='Prevalence baseline')]
primary.pivot(index='model',columns='features',values='AUC').plot.barh(figsize=(9,5),xlim=(0,1))
plt.axvline(.5,color='grey',linestyle='--');plt.title('Mean held-out ROC-AUC');plt.tight_layout()
plt.savefig(out/'auc_comparison.png',dpi=160);plt.show()
predictions=pd.DataFrame(predrows)
# One repetition for descriptive ROC/calibration plots; do not pool repeated records.
view=predictions[(predictions.version=='cleaned') & (predictions.features=='Extended') & (predictions.repeat==1)]
fig,axes=plt.subplots(1,2,figsize=(13,5))
for name in models:
 g=view[view.model==name]
 RocCurveDisplay.from_predictions(g.target,g.probability,name=name,ax=axes[0])
 CalibrationDisplay.from_predictions(g.target,g.probability,n_bins=5,strategy='quantile',name=name,ax=axes[1])
axes[0].set_title('ROC: first repetition, extended inputs')
axes[1].set_title('Calibration: first repetition, extended inputs')
plt.tight_layout();plt.savefig(out/'roc_calibration.png',dpi=160);plt.show()
print('Saved results:',out)
print('AUC: higher is better. Brier: lower is better. Sensitivity: abnormal cases detected.')

## Interpretation and next experiments
- Majority prediction already achieves 66.7% accuracy and zero sensitivity. Never use accuracy alone.
- The earlier three-model pilot found extended-model AUC about 0.64–0.66, with only 25–30% sensitivity at 0.5. Your additional-method results are exploratory and may vary with package versions.
- Lowering the threshold may improve sensitivity but produces more false positives. Choose thresholds inside inner training validation, not using held-out labels.
- For a full paper: establish novelty, verify anomalies, prespecify nested tuning/threshold selection, quantify paired uncertainty and assess external validity. No guarantee of higher performance or conference acceptance.

Outputs: per-repeat metrics, mean metrics, every held-out probability with fold assignment, cleaned selected data, dictionary, audit, run metadata, and figures. Keep the original data unchanged.

## 8. Small neural network and training-only threshold selection — added experiment
This section compares **seven classifiers**, including a small MLP, on cleaned basic and extended data. The original experiments above remain unchanged. Results go into `mlp_threshold_experiment` within the current run folder.

The MLP uses 16 and 8 hidden units, L2 regularization and early stopping (no dropout). The stopping split is drawn only from each fit's training data; outer test participants are never used. Preprocessing is fitted on that training data, including its internal stopping subset. The internal stopping score is not a generalization estimate.

For each outer fold, three-fold inner cross-validation selects a threshold targeting **80% sensitivity**. The selected threshold is then evaluated on the untouched outer fold. This target is exploratory; actual held-out sensitivity may be lower, and false positives may increase. No hyperparameter tuning or statistically significant improvement is claimed. CPU works; this section performs more fits than the earlier pilot and takes longer.

All methods use identical outer splits. Compare default and selected thresholds; their AUCs are identical because only the operating threshold changes. Repeated folds are not independent samples for significance testing.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve

# Small MLP: two hidden layers, L2 regularization, and internal early stopping.
# sklearn early stopping monitors validation accuracy. No dropout is used.
models['Small MLP'] = lambda: MLPClassifier(
    hidden_layer_sizes=(16, 8), activation='relu', solver='adam',
    alpha=0.01, batch_size=32, learning_rate_init=0.001,
    early_stopping=True, validation_fraction=0.2,
    n_iter_no_change=30, max_iter=1000, random_state=2026)
TARGET_SENSITIVITY = 0.80  # exploratory operating target; not a clinical recommendation
THRESHOLD_SEEDS = SEEDS
threshold_out = out/'mlp_threshold_experiment'
threshold_out.mkdir(parents=True, exist_ok=True)

def make_estimator(factory):
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), factory())

def choose_threshold(y_true, probabilities, target):
    fpr, tpr, thresholds = roc_curve(y_true, probabilities)
    valid = np.flatnonzero((tpr >= target) & np.isfinite(thresholds))
    # Among feasible thresholds choose minimum false-positive rate; break ties
    # by the largest threshold. Selection uses inner out-of-fold labels only.
    best_fpr = fpr[valid].min()
    ties = valid[np.isclose(fpr[valid], best_fpr)]
    return float(thresholds[ties].max())

threshold_predictions, threshold_folds, neural_training = [], [], []
for feature_set, cols in [('Basic', basic), ('Extended', extended)]:
    X = clean[cols]
    for name, factory in models.items():
        if name == 'Prevalence baseline':
            continue
        for repeat, seed in enumerate(THRESHOLD_SEEDS, 1):
            outer = StratifiedKFold(5, shuffle=True, random_state=seed)
            for fold, (train, test) in enumerate(outer.split(X, y), 1):
                inner = StratifiedKFold(3, shuffle=True, random_state=seed+fold)
                inner_p = cross_val_predict(make_estimator(factory), X.iloc[train], y[train],
                    cv=inner, method='predict_proba', n_jobs=1)[:, 1]
                threshold = choose_threshold(y[train], inner_p, TARGET_SENSITIVITY)
                estimator = make_estimator(factory)
                estimator.fit(X.iloc[train], y[train])
                probabilities = estimator.predict_proba(X.iloc[test])[:, 1]
                threshold_folds.append(dict(features=feature_set, model=name, repeat=repeat,
                    fold=fold, threshold=threshold,
                    inner_sensitivity=float((inner_p[y[train]==1] >= threshold).mean())))
                if name == 'Small MLP':
                    fitted = estimator.steps[-1][1]
                    neural_training.append(dict(features=feature_set, repeat=repeat,
                        fold=fold, epochs=fitted.n_iter_, reached_max_iter=fitted.n_iter_>=1000))
                for i, probability in zip(test, probabilities):
                    threshold_predictions.append(dict(features=feature_set, model=name,
                        repeat=repeat, fold=fold, row=int(i+1), target=int(y[i]),
                        probability=float(probability), threshold=threshold,
                        prediction_default=int(probability>=0.5),
                        prediction_selected=int(probability>=threshold)))
            print(feature_set, name, 'repeat', repeat, 'complete', flush=True)

threshold_predictions = pd.DataFrame(threshold_predictions)
threshold_predictions.to_csv(threshold_out/'held_out_predictions.csv', index=False)
pd.DataFrame(threshold_folds).to_csv(threshold_out/'selected_thresholds.csv', index=False)
pd.DataFrame(neural_training).to_csv(threshold_out/'mlp_training.csv', index=False)
threshold_metrics=[]
for (features, name, repeat), g in threshold_predictions.groupby(['features','model','repeat']):
    for operating_point, column in [('Default 0.5','prediction_default'),
                                    ('Inner-selected sensitivity target','prediction_selected')]:
        tn, fp, fn, tp = confusion_matrix(g.target, g[column], labels=[0,1]).ravel()
        threshold_metrics.append(dict(features=features, model=name, repeat=repeat,
            operating_point=operating_point, AUC=roc_auc_score(g.target,g.probability),
            AP=average_precision_score(g.target,g.probability),
            accuracy=accuracy_score(g.target,g[column]), sensitivity=tp/(tp+fn),
            specificity=tn/(tn+fp), precision=tp/(tp+fp) if tp+fp else 0.0,
            Brier=brier_score_loss(g.target,g.probability)))
threshold_metrics=pd.DataFrame(threshold_metrics)
threshold_metrics.to_csv(threshold_out/'metrics_by_repeat.csv', index=False)
threshold_summary=threshold_metrics.groupby(['features','model','operating_point'])[
    ['AUC','AP','accuracy','sensitivity','specificity','precision','Brier']].mean().reset_index()
threshold_summary.to_csv(threshold_out/'summary.csv',index=False)
(threshold_out/'protocol.json').write_text(json.dumps(dict(target_sensitivity=TARGET_SENSITIVITY,
    outer_folds=5,inner_folds=3,seeds=THRESHOLD_SEEDS,version='cleaned',
    tuning='Fixed configurations; threshold selection only',
    mlp= models['Small MLP']().get_params()),indent=2))
display(threshold_summary.round(3))
print('New results saved:',threshold_out)


## 9. What to send for review
Share the new run folder. Review `mlp_threshold_experiment/summary.csv`, `metrics_by_repeat.csv`, and `selected_thresholds.csv`. Check whether any sensitivity gain comes with an acceptable specificity trade-off. Neural network superiority is not assumed. These experiments are exploratory follow-up on a previously inspected dataset; external validation remains important.

## 10. Focused biomarker analysis — RUN THIS SECTION ONLY
This cell is self-contained: mount Drive and run it without repeating Sections 1–8. CPU is sufficient. It saves a new `runs/biomarkers_TIMESTAMP` folder and preserves all previous results.

**Question:** which biomarker groups add predictive information beyond basic measurements? Compare basic inputs with lipids, hs-CRP, hemoglobin, and all biomarkers, using fixed logistic regression and gradient boosting. The primary comparison is all biomarkers versus basic; group-specific comparisons are exploratory. These groups were chosen after reviewing earlier results, so this is not a preregistered confirmation.

Cleaning follows the existing documented procedure. Same five-fold outer splits and three seeds; fold-local imputation/scaling. No tuning, target changes, or feature selection. This measures added predictive value, not causal effects or independent biomarker effects. Hemoglobin is not HbA1c.

**Uncertainty:** 1,000 stratified paired participant bootstrap samples of held-out predictions. All predictions for a participant stay together across repetitions. The displayed 95% percentile intervals describe sampling uncertainty **conditional on the fitted models**; they exclude model-refitting variability and do not fully account for dependence induced by overlapping training folds. They are exploratory, unadjusted for multiple comparisons, and must not be treated as confirmatory significance tests. Independent validation or a full refitting analysis is still needed.

No accuracy improvement or novel conference contribution is assumed. A separate literature review remains necessary.

In [ ]:
# Standalone section: earlier experiment cells do not need to be rerun.
from pathlib import Path
from datetime import datetime
import json, hashlib, sys, subprocess
try:
    import pyreadstat
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyreadstat==1.3.6'])
    import pyreadstat
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/UAE Healthcare Prediction - Colab')
except ImportError:
    PROJECT = Path.cwd()  # Place S1_File.sav in the current folder for a local run

BIO_SEEDS = [2026, 2027, 2028]
BOOTSTRAPS = 1000
bio_out = PROJECT/'runs'/('biomarkers_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
bio_out.mkdir(parents=True, exist_ok=False)
raw, metadata = pyreadstat.read_sav(PROJECT/'S1_File.sav')
frame = raw.copy()
frame['BMI'] = raw.Weight/(raw.Height/100)**2
frame.loc[raw.DBP < 20, 'DBP'] = np.nan
labels = frame['Ab.Gly'].astype(int).to_numpy()
assert len(frame) == 555 and raw.Code.nunique() == 555
basic_cols = ['Age','Relative','BMI','WC','SBP','DBP']
lipids = ['Trigl','Chol','LDL','HDL']
feature_sets = {
    'Basic': basic_cols,
    'Basic + lipids': basic_cols + lipids,
    'Basic + hs-CRP': basic_cols + ['Hs.CRP'],
    'Basic + hemoglobin': basic_cols + ['Hb'],
    'Basic + all biomarkers': basic_cols + lipids + ['Hs.CRP','Hb']
}
model_factories = {
    'Logistic regression':lambda:LogisticRegression(C=1,max_iter=2000),
    'Gradient boosting':lambda:GradientBoostingClassifier(n_estimators=100,
        learning_rate=.05,max_depth=2,min_samples_leaf=10,random_state=2026)
}
records, scores = [], {}
for model_name, factory in model_factories.items():
    for set_name, columns in feature_sets.items():
        matrix = np.zeros((len(BIO_SEEDS),len(frame)))
        for repetition, seed in enumerate(BIO_SEEDS):
            cv = StratifiedKFold(5,shuffle=True,random_state=seed)
            for fold,(train,test) in enumerate(cv.split(frame,labels),1):
                estimator=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),factory())
                estimator.fit(frame.iloc[train][columns],labels[train])
                probabilities=estimator.predict_proba(frame.iloc[test][columns])[:,1]
                matrix[repetition,test]=probabilities
                records.extend(dict(model=model_name,features=set_name,repeat=repetition+1,
                    fold=fold,row=int(i+1),target=int(labels[i]),probability=float(p))
                    for i,p in zip(test,probabilities))
        scores[(model_name,set_name)]=matrix
        print(model_name, set_name, 'finished',flush=True)

predictions=pd.DataFrame(records)
predictions.to_csv(bio_out/'held_out_predictions.csv',index=False)
metric_rows=[]
for (model_name,set_name),matrix in scores.items():
    for repetition,p in enumerate(matrix,1):
        tn,fp,fn,tp=confusion_matrix(labels,p>=.5,labels=[0,1]).ravel()
        metric_rows.append(dict(model=model_name,features=set_name,repeat=repetition,
            AUC=roc_auc_score(labels,p),AP=average_precision_score(labels,p),
            Brier=brier_score_loss(labels,p),sensitivity=tp/(tp+fn),specificity=tn/(tn+fp)))
metrics=pd.DataFrame(metric_rows)
metrics.to_csv(bio_out/'metrics_by_repeat.csv',index=False)
summary=metrics.groupby(['model','features'])[['AUC','AP','Brier','sensitivity','specificity']].mean().reset_index()
summary.to_csv(bio_out/'summary.csv',index=False)

# Conditional paired bootstrap: sample participant rows, keeping the same row
# across repetitions, feature sets and methods. Models are NOT refitted.
rng=np.random.default_rng(2026)
positive=np.flatnonzero(labels==1); negative=np.flatnonzero(labels==0)
bootstrap_rows=[np.concatenate([rng.choice(negative,len(negative),replace=True),
                               rng.choice(positive,len(positive),replace=True)])
                for _ in range(BOOTSTRAPS)]
uncertainty=[]
for model_name in model_factories:
    base=scores[(model_name,'Basic')]
    base_auc=np.array([np.mean([roc_auc_score(labels[idx],p[idx]) for p in base])
                       for idx in bootstrap_rows])
    base_brier=np.array([np.mean((base[:,idx]-labels[idx])**2) for idx in bootstrap_rows])
    for set_name in feature_sets:
        current=scores[(model_name,set_name)]
        if set_name=='Basic':
            auc_values=base_auc; brier_values=base_brier
        else:
            auc_values=np.array([np.mean([roc_auc_score(labels[idx],p[idx]) for p in current])
                                 for idx in bootstrap_rows])
            brier_values=np.array([np.mean((current[:,idx]-labels[idx])**2) for idx in bootstrap_rows])
        delta_auc=auc_values-base_auc
        # Positive Brier reduction indicates improvement.
        delta_brier=base_brier-brier_values
        point_auc=np.mean([roc_auc_score(labels,p) for p in current])
        baseline_auc=np.mean([roc_auc_score(labels,p) for p in base])
        uncertainty.append(dict(model=model_name,features=set_name,AUC=point_auc,
            conditional_AUC_low=np.quantile(auc_values,.025),conditional_AUC_high=np.quantile(auc_values,.975),
            delta_AUC=point_auc-baseline_auc,
            conditional_delta_AUC_low=np.quantile(delta_auc,.025),
            conditional_delta_AUC_high=np.quantile(delta_auc,.975),
            Brier_reduction=float(np.mean((base-labels)**2)-np.mean((current-labels)**2)),
            conditional_Brier_reduction_low=np.quantile(delta_brier,.025),
            conditional_Brier_reduction_high=np.quantile(delta_brier,.975)))
    print('Uncertainty estimates complete:',model_name,flush=True)
uncertainty=pd.DataFrame(uncertainty)
uncertainty.to_csv(bio_out/'conditional_uncertainty.csv',index=False)
protocol=dict(seeds=BIO_SEEDS,folds=5,bootstrap_samples=BOOTSTRAPS,
    bootstrap='Stratified paired participant bootstrap of fixed out-of-fold predictions; no refitting',
    primary_comparison='Basic + all biomarkers versus Basic, separately for each model',
    secondary_comparisons='Individual biomarker groups versus Basic; exploratory, unadjusted intervals',
    feature_sets=feature_sets,threshold=.5,sklearn=sklearn.__version__,
    input_sha256=hashlib.sha256((PROJECT/'S1_File.sav').read_bytes()).hexdigest())
(bio_out/'protocol.json').write_text(json.dumps(protocol,indent=2))
display(summary.round(3))
display(uncertainty.round(3))
print('Saved to:',bio_out)
print('Intervals are conditional on fitted predictions, not full training uncertainty or confirmatory significance tests.')


### After running
The key files are `summary.csv`, `conditional_uncertainty.csv`, and `metrics_by_repeat.csv` in the new biomarker folder. Positive delta AUC or positive Brier reduction indicates an improvement over basic inputs. If an interval spans zero, the comparison is inconclusive under this limited analysis; an interval excluding zero still does not establish a confirmatory finding. Share that run folder for interpretation.